In [1]:
import pandas as pd
import numpy as np
import warnings
import random
import re
import os
import json
warnings.filterwarnings("ignore")

from tqdm import tqdm
from copy import deepcopy

PREFIX = "../../data/1105"
data_PREFIX = "../../data/1022"

In [2]:
# system_msg = "You are a behavioral neurologist. Follow these conditions when generating the answer:\n1. Your task is to answer the question by basing your response solely on the 'Patient' section. 'Diagnosis' section is only provided for your reference. It is not patient history. Use it only to verify your analysis.\n2. You are allowed to use the 'Diagnosis' section of the input indirectly to guide your answers. But, do not reference any details from the 'Diagnosis' section in your responses.\n3. Do not use terms like 'clinician', 'clinician's assessment', 'clinical diagnosis', 'deemed by a clinician' or any other phrases indicating a confirmed diagnosis. Instead use phrases such as 'I diagnose', 'I assess', and so on indicating you are the one making the assessment.\n4. Answer responsibly, avoiding overconfidence, and encourage the user to consult a healthcare professional for advice.\n5. Your responses should not include any recommendations.\n6. When any information is missing, assume it was not collected.\n7. Please verify your answers with the 'Diagnosis' section in the input."
# print(system_msg)

In [2]:
def get_diagnosis_dict(datadict_path):
    with open(datadict_path, 'r', encoding='utf-8') as f:
        datadict = json.load(f)
        
    cog_var = []
    oth = []
    for variable, info in datadict.items():
        if info['FORM ID'] == 'D1':
            desc = info['Description']
            if ('normal cognition' in desc.lower() or 'mci' in desc.lower() or 'cognitive status' in desc.lower() or 'met criteria for' in desc.lower() or 'during uds follow-up' in desc.lower()) and ('Incident' not in desc):
                # print(desc)
                cog_var.append(desc)
            else:
                oth.append(desc)
                
    diagnosis_dict = {'Cognitive Status': datadict['NACCUDSD']['Values']}
    diagnosis_dict['Cognitive Status'].pop('2')
    
    return diagnosis_dict, oth, cog_var

In [3]:
datadict_path = "/projectnb/vkolagrp/datasets/NACC/data_dictionaries/NACC_dictionary.json"
diagnosis_dict, oth, cog_var = get_diagnosis_dict(datadict_path)

In [4]:
diagnosis_dict

{'Cognitive Status': {'1': 'Normal cognition', '3': 'MCI', '4': 'Dementia'}}

## Start cleaning generated responses

In [5]:
import json

# data = []
# with open(f"{PREFIX}/diagnosis/nacc_summaries_json_output_1.json", "r") as file:
#     for line in file:
#         try:
#             line = json.loads(line)
#             if line['NACCUDSD'] != 2:
#                 data.append(line)
#         except json.JSONDecodeError as e:
#             print(f"Error decoding JSON: {e}")

data = []
with open(f"{PREFIX}/diagnosis/nacc_summaries_json_output_2.json", "r") as file:
    for line in file:
        try:
            line = json.loads(line)
            if line['NACCUDSD'] != 2:
                data.append(line)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

print(f'Length of data Before cleaning: {len(data)}')

Length of data Before cleaning: 105669


In [7]:
print(data[0]['NACCUDSD'])

4


In [6]:
print(f'Length of data before cleaning: {len(data)}')

Length of data before cleaning: 105669


In [ ]:
for i in range(len(data)):
    answer = f"""Answer: {list(x.keys())[0]} - {diagnosis_dict['Cognitive Status'][str(x['Cognitive Status']['value'])]}
Explanation: {x['Cognitive Status']['explanation']}"""
    print(answer)
    # print(data[i].keys())
    # print(json.loads(data[i]['answer'])['Cognitive Status']['explanation'])

In [8]:
ids = set()
ids_list = list()
question_set = set()

for val in data:
    ids.add(val['NACCID'])
    ids_list.append(val['NACCID'])
    question_set.add(val['instruction'])

In [9]:
question_set

{'Provide the Cognitive status at UDS visit for a patient presenting with the following information. Please Report them in JSON format according to the following dictionary.\n{\n    "Cognitive Status": {\n        "1": "Normal cognition",\n        "3": "MCI",\n        "4": "Dementia"\n    }\n}'}

In [10]:
print(len(ids))

29185


#### Identify answers with mentions of clinician's assessment or diagnosis section

In [11]:
clinician_diagnosis_section_answers = {}
j = 0
for i, entry in enumerate(data):
    json_entry = json.loads(entry['answer'])
    if ("clinician" in json_entry['Cognitive Status']['explanation'].lower()) or ("clinical diagnosis" in json_entry['Cognitive Status']['explanation'].lower()) or ("diagnosis section" in json_entry['Cognitive Status']['explanation'].lower()):
        clinician_diagnosis_section_answers[i] = entry

In [12]:
len(clinician_diagnosis_section_answers)

60

#### Identify answers with wrong diagnosis

In [13]:
wrong_diagnosis = {}
mismatch = 0
for i, entry in enumerate(data):
    json_entry = json.loads(entry['answer'])
    if json_entry['Cognitive Status']['value'] != entry['NACCUDSD']:
        # print(json_entry['Cognitive Status'])
        # print(entry['NACCUDSD'])
        mismatch += 1
        wrong_diagnosis[i] = entry
        
mismatch

92

In [16]:
wrong_diagnosis[1009]

{'index': 1039,
 'NACCID': 'NACC271973',
 'NACCVNUM': 3,
 'NACCMNUM': nan,
 'NACCNMRI': 0,
 'NACCMRSA': 0,
 'NACCADC': 2578,
 'VISITMO': 3,
 'VISITDAY': 5,
 'VISITYR': 2015,
 'MRIMO': nan,
 'MRIDY': nan,
 'MRIYR': nan,
 'NACCUDSD': 1,
 'NACCETPR': 88,
 'CDRGLOB': 0.5,
 'instruction': 'Provide the Cognitive status at UDS visit for a patient presenting with the following information. Please Report them in JSON format according to the following dictionary.\n{\n    "Cognitive Status": {\n        "1": "Normal cognition",\n        "3": "MCI",\n        "4": "Dementia"\n    }\n}',
 'patient_summary': '```json\n{\n\t"Subject Demographics": {\n\t\t"Living situation": "Lives with spouse or partner",\n\t\t"Primary language": "Other primary language (specify)",\n\t\t"Primary language, other- specify": "German",\n\t\t"Level of independence": "Completely dependent",\n\t\t"Race": "White",\n\t\t"Second race": "None reported",\n\t\t"Hispanic/Latino ethnicity": "No",\n\t\t"Primary reason for coming to AD

In [14]:
len(wrong_diagnosis) 

92

## Identify the indices of the responses with repeated sentences

In [17]:
import nltk
from collections import defaultdict

# Ensure you have the Punkt tokenizer models downloaded
nltk.download('punkt') #, download_dir='/projectnb/vkolagrp/skowshik/.cache/punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     /usr3/graduate/skowshik/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [18]:
def find_highly_repeated_sentences(responses):
    nums = ['1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.', '9.', '*']
    highly_repeated = defaultdict(list)
    
    for i, response in enumerate(responses):
        # Tokenize the response into sentences
        sentences = nltk.sent_tokenize(response)
        sentence_counts = defaultdict(int)
        
        # Count each sentence
        for sentence in sentences:
            if sentence in nums:
                continue
            sentence_counts[sentence] += 1
        
        # Check for sentences with more than two repetitions
        for sentence, count in sentence_counts.items():
            if count > 4:
                highly_repeated[i].append((sentence, count))

    return highly_repeated

def modify_response_to_keep_single_repeated(responses):
    modified_responses = {}
    highly_repeated_indices = defaultdict(list)
    
    for i, response in enumerate(responses):
        # Tokenize the response into sentences
        sentences = nltk.sent_tokenize(response)
        sentence_counts = defaultdict(int)
        
        # Count each sentence
        for sentence in sentences:
            sentence_counts[sentence] += 1
            
        # Check for sentences with more than two repetitions
        for sentence, count in sentence_counts.items():
            if count > 4:
                highly_repeated_indices[i].append((sentence, count))

        # Find sentences repeated more than four times
        highly_repeated = {sentence for sentence, count in sentence_counts.items() if count > 4}

        # Reconstruct the response keeping only one instance of each highly repeated sentence
        modified_response = []
        seen = set()
        for sentence in sentences:
            if sentence in highly_repeated:
                if sentence not in seen:
                    modified_response.append(sentence)
                    seen.add(sentence)
            else:
                modified_response.append(sentence)

        # Join modified response back into a single string
        if i in highly_repeated_indices:
            modified_responses[i] = ' '.join(modified_response)
        
    return modified_responses, highly_repeated_indices

def get_repeated_indices(key, data_list):
    # new_data = deepcopy(data_list)
    
    responses = []
    for i in range(len(data_list)):
        responses.append(data_list[i][key])

    repeat_indices = []
    # Find sentences repeated more than 4 times
    highly_repeated_indices = find_highly_repeated_sentences(responses)
    # modified_responses_dict, highly_repeated_indices = modify_response_to_keep_single_repeated(responses)
    
    # Print the results
    for index, repeat in highly_repeated_indices.items():
        repeat_indices.append(index)
    
    return highly_repeated_indices, repeat_indices
    
    # # Print the results
    # for index, modified_response in modified_responses_dict.items():
    #     new_data[index][key] = modified_response
    #     # print(new_data[i][key])
    #     # print(index, "modified")

    # return new_data, repeat_indices


In [19]:
# patient_modified_data, patient_repeat_indices = get_repeated_indices('patient_LLAMA_SUMMARY', data)
# diag_modified_data, diag_repeat_indices = get_repeated_indices('diag_LLAMA_SUMMARY', data)
repeats, diag_repeat_indices = get_repeated_indices('answer', data)

In [20]:
print(f"Number of patient responses repeated {len(diag_repeat_indices)}")
# print(f"Number of diagnosis responses repeated {len(diag_repeat_indices)}")

Number of patient responses repeated 0


In [24]:
repeats

defaultdict(list, {})

In [25]:
# ind = 4
# print(data[diag_repeat_indices[ind]]['instruction'], data[diag_repeat_indices[ind]]['NACCID'])
# print(data[diag_repeat_indices[ind]]['answer'])

## Clean the responses

In [26]:
# Delete the responses by ordering them in descending order
def delete_by_indices(lst, indices):
    # Sort the indices in descending order to avoid index shift issues
    for index in sorted(indices, reverse=True):
        if index < len(lst):
            del lst[index]

In [27]:
indices_to_delete = []
for ind in diag_repeat_indices:
    indices_to_delete.append(ind)

In [28]:
delete_by_indices(data, indices_to_delete)
delete_by_indices(data, wrong_diagnosis)
delete_by_indices(data, clinician_diagnosis_section_answers)

In [29]:
print(f'Length of data After cleaning: {len(data)}')

Length of data After cleaning: 585849


#### Add MRI paths to the patient summary

In [154]:
# def get_mri_or_emb_dict(val):
#     mris_found = defaultdict(int)
#     if val == 'mri':
#         data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped'
#         save_path = 'nacc_mri_paths.json'
#         extension = '.nii'
#     else:
#         data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/swin_emb'
#         save_path = 'nacc_emb_paths.json'
#         extension = '.npy'
#     j = 6
#     def extract_id(name):
#         # This regex matches any characters following "fname-" up until the first occurrence of "_mo"
#         match = re.search(r'fname-(.*?)_mo', name)
#         if match:
#             return match.group(1)    # Returns the captured group which is the part of the string you need
#         else:
#             match = re.search(r'fname-(.*?)_nodate', name)
#             if match:
#                 return match.group(1) 
#         return None

#     mri_dict = {}
#     # for dirpath, dirs, files in tqdm(os.walk('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped')): 
#     for dirpath, dirs, files in tqdm(os.walk(data_path)): 
#         if len(files) == 0:
#             continue
#         for filename in files:
#             fname = os.path.join(dirpath, filename)
#             path_list = fname.split('/')
#             naccid = path_list[j+1]
#             # print(path_list)
#             zip_name = extract_id(path_list[j+2])
#             modality = path_list[j+3]
#             acquisition = path_list[j+4]
#             mri_name = path_list[j+5]
#             if mri_name in mris_found:
#                 continue
#             mris_found[mri_name] += 1
#             # print(naccid, zip_name, modality, acquisition, mri_name)
            
#             if modality == 'Unknown' or (not mri_name.endswith(extension)):
#                 continue
#             # if naccid not in mri_dict:
#             #     mri_dict[naccid] = {}
            
#             if zip_name + 'ni' not in mri_dict:
#                 mri_dict[zip_name + 'ni'] = {}
            
#             mri_dict[zip_name + 'ni'][modality] = {"acquisition": acquisition, "mri_name": fname}
            
                
                

#     # with open(save_path, "w") as outfile:
#     #     json.dump(mri_dict, outfile, indent=4, sort_keys=False)
        
#     return mri_dict, mris_found

In [155]:
# emb_dict, mris_found = get_mri_or_emb_dict(val='emb')

In [156]:
# modified_data = deepcopy(data)
# for i, entry in enumerate(modified_data):
#     zip_name = nacc.loc[nacc['NACCID'] == entry['NACCID']]['NACCMRFI'].iloc[0]
#     if isinstance(zip_name, str):
#         mod_zip_name = zip_name.replace(".zip", "ni")
#         if mod_zip_name in emb_dict:
#             mri_text = ''
#             for seq, info in emb_dict[mod_zip_name].items():
#                 if "diffusion" in seq.lower():
#                     continue
#                 mri_text += f"{seq} MRI image: {info['mri_name']} "
            
#             print(mri_text)
#             modified_data[i]['patient_LLAMA_SUMMARY'] += f"\n\n**MRI imaging**:\n\n{mri_text}"
#             # print(modified_data[i]['patient_LLAMA_SUMMARY'])
#             # break

## Save train and test datasets

In [30]:
nacc_np = pd.read_csv(f'/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_neuropath_vnum_unique_nacc65.csv')
ct_path = f"{PREFIX}/ct/ct_data.json"       
with open(ct_path, 'r', encoding='utf-8') as json_file:
    data_ct = json.load(json_file)
    
eeg_path = f"{PREFIX}/eeg/eeg_data.json"       
with open(eeg_path, 'r', encoding='utf-8') as json_file:
    data_eeg = json.load(json_file)

In [31]:
len(nacc_np)

34240

In [32]:
new_data = {}
i = 1
for entry in data:
    new_data[f"NACC_{i}"] = entry
    i += 1

In [33]:
data_train = {}
data_test = {}
train = 1
test = 1
np_ids = set(list(nacc_np['NACCID']))
for entry in data:
    if (entry['NACCID'] in np_ids):
        data_test[f"NACC_{test}"] = entry
        test += 1
    else:
        data_train[f"NACC_{train}"] = entry
        train += 1
        
data_train.update(data_ct)
data_train.update(data_eeg)

In [34]:
print(f"Train split: {len(data_train)}")
print(f"Test split: {len(data_test)}")

Train split: 453551
Test split: 134493


In [ ]:
json_name_train = f"{PREFIX}/train_data/train.json"
with open(json_name_train, 'w', encoding='utf-8') as json_file:
    json.dump(data_train, json_file, indent=4, sort_keys=False)

json_name_test = f"{PREFIX}/train_data/test.json"
with open(json_name_test, 'w', encoding='utf-8') as json_file:
    json.dump(data_test, json_file, indent=4, sort_keys=False)